# Query Webull Market Price Snapshots

This notebook makes read-only Webull market-data requests and prints the raw response plus a small interpretation summary. It is designed to reveal response shapes, missing fields, HTTP errors, and SDK/API issues without stopping at the first failed symbol.

Install the optional SDK first if needed: `python -m pip install -e '.[webull]'`.

In [17]:
import json
import os
from pathlib import Path


def load_dotenv_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip()
        value = value.strip().strip("'").strip('\"')
        os.environ.setdefault(name, value)


load_dotenv_file(Path.cwd() / ".env")
required = ("WEBULL_APP_KEY", "WEBULL_APP_SECRET")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing environment variable(s): {', '.join(missing)}")

region = os.environ.get("WEBULL_REGION", "us")
endpoint = os.environ.get("WEBULL_API_ENDPOINT", "api.sandbox.webull.com")
market = os.environ.get("WEBULL_MARKET", "US").upper()
symbols = [
    symbol.strip().upper()
    for symbol in os.environ.get("WEBULL_SYMBOLS", "MSFT").split(",")
    if symbol.strip()
]

print(f"Using Webull region={region!r}, endpoint={endpoint!r}, market={market!r}")
print(f"Symbols: {symbols}")

Using Webull region='us', endpoint='api.sandbox.webull.com', market='US'
Symbols: ['MSFT']


In [18]:
from webull.core.client import ApiClient
from webull.data.common.category import Category
from webull.data.data_client import DataClient

category_name = f"{market}_STOCK"
try:
    category = getattr(Category, category_name).name
except AttributeError as exc:
    raise RuntimeError(f"Unsupported Webull market category: {category_name}") from exc

api_client = ApiClient(
    os.environ["WEBULL_APP_KEY"],
    os.environ["WEBULL_APP_SECRET"],
    region,
)
api_client.add_endpoint(region, endpoint)
data_client = DataClient(api_client)
print(f"Resolved category: {category!r}")

139463679153984 2026-09-23 04:02:22,159 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
139463679153984 2026-09-23 04:02:22,159 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
139463679153984 2026-09-23 04:02:22,159 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
139463679153984 2026-09-23 04:02:22,159 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
139463679153984 2026-09-23 04:02:22,159 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
Resolved category: 'US_STOCK'


In [19]:
def response_json(response):
    try:
        return response.json()
    except (ValueError, TypeError):
        return None


def extract_summary(payload, symbol):
    values = payload.get("data", payload) if isinstance(payload, dict) else payload
    if isinstance(values, list):
        values = next(
            (item for item in values if isinstance(item, dict) and item.get("symbol") == symbol),
            values[0] if values else {},
        )
    if not isinstance(values, dict):
        return {"payload_shape": type(values).__name__, "last_price": None, "market_status": None}
    price_keys = ("last_price", "lastPrice", "latest", "latest_price", "close")
    status_keys = ("market_status", "marketStatus", "is_market_open")
    price_key = next((key for key in price_keys if values.get(key) not in (None, "")), None)
    status_key = next((key for key in status_keys if key in values), None)
    return {
        "payload_shape": "dict",
        "top_level_keys": sorted(values.keys()),
        "last_price": values.get(price_key) if price_key else None,
        "price_field": price_key,
        "market_status": values.get(status_key) if status_key else None,
        "status_field": status_key,
    }


results = {}
for symbol in symbols:
    print(f"\n{'=' * 80}\n{symbol}")
    try:
        response = data_client.market_data.get_snapshot(
            symbol,
            category,
            extend_hour_required=False,
            overnight_required=True,
        )
        payload = response_json(response)
        results[symbol] = {"response": response, "payload": payload}
        print(f"HTTP status: {response.status_code}")
        print("Raw response text:")
        print(response.text or "<empty>")
        if response.status_code != 200:
            print(f"WARNING: non-200 response for {symbol}")
        if payload is None:
            print("WARNING: response body is not valid JSON")
        else:
            print("Parsed JSON:")
            print(json.dumps(payload, indent=2, sort_keys=True, default=str))
            # summary = extract_summary(payload, symbol)
            # print("Extracted summary:")
            # print(json.dumps(summary, indent=2, sort_keys=True, default=str))
            # if summary.get("last_price") in (None, ""):
            #     print("WARNING: no likely last-price field was found")
            # if summary.get("payload_shape") != "dict":
            #     print("WARNING: unexpected payload shape")
    except Exception as exc:
        results[symbol] = {"exception": exc}
        print(f"ERROR: {type(exc).__name__}: {exc}")

# print("\nRequest summary:")
# for symbol, result in results.items():
#     if "exception" in result:
#         print(f"{symbol}: exception={type(result['exception']).__name__}")
#     else:
#         response = result["response"]
#         print(f"{symbol}: HTTP {response.status_code}")


MSFT
HTTP status: 200
Raw response text:
[{"symbol":"MSFT","price":"498.0000","open":"507.3200","high":"508.5000","low":"493.6500","volume":"21652838","change":"-3.610000","close":"498.0000","instrument_id":"913323997","pre_close":"501.610000","change_ratio":"-0.007197","last_trade_time":1790107201010,"ask":"501.9100","ask_size":"15","bid":"499.9500","bid_size":"7","quote_time":1790121600001,"ovn_price":"500.10","ovn_high":"500.86","ovn_low":"499.19","ovn_volume":"22548","ovn_change":"2.1000","ovn_change_ratio":"0.004217","ovn_last_trade_time":1790135153565,"ovn_ask":"500.19","ovn_ask_size":"40","ovn_bid":"500.11","ovn_bid_size":"20","ovn_quote_time":1790135181290,"pb_ratio":"8.360657","ps_ratio":"11.145902","pe_ratio":"27.750380","market_value":"3697921654518.0000","neg_market_value":"3693947198190.0000","yield":"0.007871","total_shares":"7425545491","out_standing_shares":"7417564655","fifty_two_wk_high":"549.201343","fifty_two_wk_low":"348.543867","turnover":"0.002150","eps":"17.945